In [1]:
%load_ext IPython.extensions.autoreload
%autoreload 2
from neuronsim.simulation import *
from neuronsim.params import *
import numpy as np
import matplotlib.pyplot as plt
import pickle as pkl
import gzip
import os
plt.rcParams.update({
    "text.usetex": True
})

In [8]:
Is = np.linspace(3.6, 4, 20)
for beta in np.arange(0, 1, 11):
    firing_rates = []
    for I in Is:
        k = 1000
        delta_g_k_ca = 0
        inhib_fraction = 0.3
        v_epsp = 1
        v_ipsp = -1.5
        params = network_C(I, k, beta, delta_g_k_ca, inhib_fraction, v_epsp, v_ipsp)
        pkl_path = f"pkls/network_C/k-{k}/beta-{beta}/delta_g_k_ca-{delta_g_k_ca}/inhib_fraction-{inhib_fraction}/v_epsp-{v_epsp}/v_ipsp-{v_ipsp}/"
        pkl_name = f"I-{I}.pkl.gzip" 
        if not os.path.exists(pkl_path):
            os.makedirs(pkl_path)
        res = run_sim(*params)
        
        with gzip.open(pkl_path + pkl_name, "wb") as f:
            pkl.dump(res, f)
            
        t, spikes, final_state = res
        _, _, network_params = params
            
        firing_rate = spikes[:,final_state.neuron_types].sum() / 100 / network_params.n / (1 - network_params.inhib_fraction)
        firing_rates.append(firing_rate)

    plt.title(f"$\beta = ${beta}")
    plt.ylabel("Mean firing rate (Hz)")
    plt.xlabel("$\\hat{I}_{max}$ (normalised)")
    plt.scatter(Is, firing_rates)
    plt.vlines([3.715, 3.75], 0, 0.5)
    plt.show()

KeyboardInterrupt: 

In [ ]:
# Setup a Gaussian window for convolutions
def gaus(x, width):
    return 1 / width / np.sqrt(2 * np.pi) * np.exp(-1 / 2 * (x / width)**2)
gaus_window = gaus(np.arange(-30,30), 10)

In [ ]:
Is = np.concat([np.linspace(3.6, 4, 20)[:-1], np.linspace(4, 4.4, 20)])[:-10]
for beta in np.arange(0, 1, 11):
    high_firing_rates = []
    low_firing_rates = []
    for I in Is:
        k = 1000
        delta_g_k_ca = 0.01
        inhib_fraction = 0.3
        v_epsp = 1
        v_ipsp = -1.5
        params = network_C(I, k, beta, delta_g_k_ca, inhib_fraction, v_epsp, v_ipsp)
        pkl_path = f"pkls/network_C/k-{k}/beta-{beta}/delta_g_k_ca-{delta_g_k_ca}/inhib_fraction-{inhib_fraction}/v_epsp-{v_epsp}/v_ipsp-{v_ipsp}/"
        pkl_name = f"I-{I}.pkl.gzip" 
        if not os.path.exists(pkl_path):
            os.makedirs(pkl_path)
        res = None
            
        #res = run_sim(*params)
        #with gzip.open(pkl_path + pkl_name, "wb") as f:
        #    pkl.dump(res, f)
        
        with gzip.open(pkl_path + pkl_name, "rb") as f:
            res = pkl.load(f)
            
        t, spikes, final_state = res
        _, _, network_params = params
        
        activity = spikes.sum(axis=1)
        # use a rolling average to smooth out noise and find bursting windows
        bursting_filter = (np.convolve(activity, gaus_window) > 0)[29:-30]
        # say somewhat arbitrarily that if we see activity windows after 50s that there exists a stable/metastable state
        # and that if there is a total period of >1s of silence after 50s that that means bursting
        firing_rate = None
        if np.any(bursting_filter[50000:]) and np.sum(~bursting_filter[50000:]) >= 1000:
            bursts = spikes[bursting_filter,]
            firing_rate = bursts[:,final_state.neuron_types].sum() / (bursts.shape[0] / 1000) / network_params.n / (1 - network_params.inhib_fraction)
            high_firing_rates.append(firing_rate)
            low_firing_rates.append(0)
        else:
            firing_rate = spikes[:,final_state.neuron_types].sum() / 100 / network_params.n / (1 - network_params.inhib_fraction)
            high_firing_rates.append(firing_rate)
            low_firing_rates.append(None)
    plt.title(f"$\\beta = {beta}")
    plt.ylabel("Mean firing rate (Hz)")
    plt.xlabel("$\\hat{I}_{max}$ (normalised)")
    plt.scatter(Is, np.array(high_firing_rates))
    plt.scatter(Is, np.array(low_firing_rates))
    plt.vlines([3.715, 3.75], 0, 0.32)
    plt.show()
            